# 03 - Feature Engineering

## Overview
This notebook covers:
- Temporal feature extraction
- Electrical feature calculation
- Load categorization
- Anomaly detection
- Rolling window features

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json

print("Imports successful")

Imports successful


In [2]:
# Setup paths and load cleaned data
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

cleaned_file = DATA_PROCESSED / 'output_cleaned.csv'
df = pd.read_csv(cleaned_file)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"Loaded {len(df):,} cleaned records")
print(f"Columns: {list(df.columns)}")

Loaded 24,400 cleaned records
Columns: ['Timestamp', 'Meter_ID', 'Zone_ID', 'Voltage_V', 'Current_A', 'Active_Power_kW', 'Reactive_Power_kW', 'Apparent_Power_kVA', 'Frequency_Hz', 'Sub_Meter_Kitchen', 'Sub_Meter_HVAC', 'Outdoor_Temp_C', 'is_outlier']


## 1. Temporal Features

In [3]:
# Extract temporal features from Timestamp
print("\n=== TEMPORAL FEATURES ===")

df['year'] = df['Timestamp'].dt.year
df['month'] = df['Timestamp'].dt.month
df['day'] = df['Timestamp'].dt.day
df['hour'] = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['week_of_year'] = df['Timestamp'].dt.isocalendar().week
df['day_of_year'] = df['Timestamp'].dt.dayofyear
df['quarter'] = df['Timestamp'].dt.quarter

# Season mapping
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df['month'].apply(get_season)

# Day of week name
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_name'] = df['day_of_week'].apply(lambda x: day_names[x])

print("[CHECK]“ Temporal features created:")
print(f"  - year, month, day, hour")
print(f"  - day_of_week, week_of_year, day_of_year")
print(f"  - quarter, season, day_name")


=== TEMPORAL FEATURES ===
[CHECK]“ Temporal features created:
  - year, month, day, hour
  - day_of_week, week_of_year, day_of_year
  - quarter, season, day_name


## 2. Peak Hour Flagging

In [4]:
# Define peak hours (typically morning 6-9 AM and evening 6-9 PM)
def is_peak_hour(hour):
    return 1 if hour in [6, 7, 8, 9, 18, 19, 20, 21] else 0

df['peak_hour_flag'] = df['hour'].apply(is_peak_hour)
peak_records = df['peak_hour_flag'].sum()

print("\n=== PEAK HOURS ===")
print(f"Peak hours flagged: {peak_records:,} records ({peak_records/len(df)*100:.1f}%)")
print(f"Peak hours: 6-9 AM, 6-9 PM")


=== PEAK HOURS ===
Peak hours flagged: 8,029 records (32.9%)
Peak hours: 6-9 AM, 6-9 PM


## 3. Electrical Features

In [5]:
# Power factor calculation
print("\n=== ELECTRICAL FEATURES ===")

if 'Apparent_Power_kVA' in df.columns:
    df['power_factor'] = np.where(
        df['Apparent_Power_kVA'] != 0,
        df['Active_Power_kW'] / df['Apparent_Power_kVA'],
        0
    )
    print(f"[CHECK]“ Power factor calculated (mean: {df['power_factor'].mean():.3f})")

# Power factor category
def categorize_pf(pf):
    if pf >= 0.95:
        return 'Excellent'
    elif pf >= 0.85:
        return 'Good'
    elif pf >= 0.70:
        return 'Fair'
    else:
        return 'Poor'

if 'power_factor' in df.columns:
    df['power_factor_category'] = df['power_factor'].apply(categorize_pf)
    print("[CHECK]“ Power factor categories assigned")


=== ELECTRICAL FEATURES ===
[CHECK]“ Power factor calculated (mean: 0.912)
[CHECK]“ Power factor categories assigned


## 4. Load Categorization

In [6]:
# Categorize load based on consumption quartiles
print("\n=== LOAD CATEGORIZATION ===")

Q1 = df['Active_Power_kW'].quantile(0.25)
Q2 = df['Active_Power_kW'].quantile(0.50)
Q3 = df['Active_Power_kW'].quantile(0.75)

print(f"Q1 (25%): {Q1:.2f} kW")
print(f"Q2 (50%): {Q2:.2f} kW")
print(f"Q3 (75%): {Q3:.2f} kW")

# Categorize
df['load_category'] = pd.cut(
    df['Active_Power_kW'],
    bins=[0, Q1, Q2, Q3, float('inf')],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print(f"\nLoad distribution:")
print(df['load_category'].value_counts().sort_index())


=== LOAD CATEGORIZATION ===
Q1 (25%): 1.18 kW
Q2 (50%): 1.76 kW
Q3 (75%): 3.02 kW

Load distribution:
load_category
Low         5838
Medium      6039
High        6036
Critical    6037
Name: count, dtype: int64


## 5. Anomaly Detection

In [7]:
# Detect anomalies using IQR method
print("\n=== ANOMALY DETECTION ===")

Q1_anom = df['Active_Power_kW'].quantile(0.25)
Q3_anom = df['Active_Power_kW'].quantile(0.75)
IQR = Q3_anom - Q1_anom
lower_bound = Q1_anom - 1.5 * IQR
upper_bound = Q3_anom + 1.5 * IQR

print(f"Lower bound: {lower_bound:.2f} kW")
print(f"Upper bound: {upper_bound:.2f} kW")

# Flag anomalies
df['is_anomaly'] = ((df['Active_Power_kW'] < lower_bound) | (df['Active_Power_kW'] > upper_bound)).astype(int)
anomaly_count = df['is_anomaly'].sum()
anomaly_pct = anomaly_count / len(df) * 100

print(f"\nAnomalies detected: {anomaly_count:,} ({anomaly_pct:.2f}%)")


=== ANOMALY DETECTION ===
Lower bound: -1.59 kW
Upper bound: 5.78 kW

Anomalies detected: 214 (0.88%)


## 6. Consumption Trends

In [8]:
# Calculate rolling averages
print("\n=== ROLLING WINDOW FEATURES ===")

# 24-hour rolling average (hourly basis)
df_sorted = df.sort_values(['Meter_ID', 'Timestamp']).reset_index(drop=True)
df_sorted['power_rolling_24h'] = df_sorted.groupby('Meter_ID')['Active_Power_kW'].transform(
    lambda x: x.rolling(window=24, min_periods=1).mean()
)

# 7-day rolling average (assuming daily readings)
df_sorted['power_rolling_7d'] = df_sorted.groupby('Meter_ID')['Active_Power_kW'].transform(
    lambda x: x.rolling(window=168, min_periods=1).mean()  # 168 hours = 7 days
)

df = df_sorted
print(f"[CHECK]“ 24-hour rolling average calculated")
print(f"[CHECK]“ 7-day rolling average calculated")


=== ROLLING WINDOW FEATURES ===
[CHECK]“ 24-hour rolling average calculated
[CHECK]“ 7-day rolling average calculated


## 7. Consumption Change Rate

In [9]:
# Calculate consumption change rate (hour-to-hour)
df['consumption_change'] = df.groupby('Meter_ID')['Active_Power_kW'].diff()
df['consumption_change_pct'] = df.groupby('Meter_ID')['Active_Power_kW'].pct_change() * 100

print("\n=== CONSUMPTION DYNAMICS ===")
print(f"Mean hour-to-hour change: {df['consumption_change'].mean():.2f} kW")
print(f"Max increase: {df['consumption_change'].max():.2f} kW")
print(f"Max decrease: {df['consumption_change'].min():.2f} kW")


=== CONSUMPTION DYNAMICS ===
Mean hour-to-hour change: 0.00 kW
Max increase: 147.11 kW
Max decrease: -147.19 kW


## 8. Is Weekend Flag

In [10]:
# Weekend flag
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
print(f"\nWeekend records: {df['is_weekend'].sum():,} ({df['is_weekend'].sum()/len(df)*100:.1f}%)")


Weekend records: 6,969 (28.6%)


## 9. Feature Summary

In [11]:
# List all engineered features
original_cols = ['Timestamp', 'Meter_ID', 'Zone', 'Active_Power_kW', 'Reactive_Power_kVAR', 'Apparent_Power_kVA']
new_features = [col for col in df.columns if col not in original_cols]

print("\n=== ENGINEERED FEATURES ===")
for i, feature in enumerate(new_features, 1):
    print(f"{i:2d}. {feature}")

print(f"\nTotal features: {len(df.columns)}")


=== ENGINEERED FEATURES ===
 1. Zone_ID
 2. Voltage_V
 3. Current_A
 4. Reactive_Power_kW
 5. Frequency_Hz
 6. Sub_Meter_Kitchen
 7. Sub_Meter_HVAC
 8. Outdoor_Temp_C
 9. is_outlier
10. year
11. month
12. day
13. hour
14. day_of_week
15. week_of_year
16. day_of_year
17. quarter
18. season
19. day_name
20. peak_hour_flag
21. power_factor
22. power_factor_category
23. load_category
24. is_anomaly
25. power_rolling_24h
26. power_rolling_7d
27. consumption_change
28. consumption_change_pct
29. is_weekend

Total features: 33


## 10. Export Featured Data

In [12]:
# Save featured data
output_file = Path(PROJECT_ROOT) / 'data' / 'processed' / 'output_featured.csv'
df.to_csv(output_file, index=False)

print(f"\n[CHECK]“ Featured data saved to: {output_file}")
print(f"  Records: {len(df):,}")
print(f"  Features: {len(df.columns)}")

# Feature engineering summary
fe_summary = {
    'timestamp': datetime.now().isoformat(),
    'total_records': len(df),
    'original_columns': len(original_cols),
    'engineered_features': len(new_features),
    'total_columns': len(df.columns),
    'engineered_feature_names': new_features
}

summary_file = Path(PROJECT_ROOT) / 'data' / 'processed' / 'feature_engineering_report.json'
with open(summary_file, 'w') as f:
    json.dump(fe_summary, f, indent=2, default=str)

print(f"[CHECK]“ Feature report saved to: {summary_file}")


[CHECK]“ Featured data saved to: C:\Smart Meter Data Systems\data\processed\output_featured.csv
  Records: 24,400
  Features: 33
[CHECK]“ Feature report saved to: C:\Smart Meter Data Systems\data\processed\feature_engineering_report.json


## 11. Summary

In [13]:
print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"[CHECK]“ Temporal features created: 9 new columns")
print(f"[CHECK]“ Electrical features calculated: power_factor")
print(f"[CHECK]“ Load categories assigned: 4 categories")
print(f"[CHECK]“ Anomalies detected: {anomaly_count:,} records")
print(f"[CHECK]“ Rolling averages computed: 24h, 7d")
print(f"[CHECK]“ Total features engineered: {len(new_features)}")
print(f"\nNext: Proceed to EDA analysis notebook")
print("="*80)


FEATURE ENGINEERING SUMMARY
[CHECK]“ Temporal features created: 9 new columns
[CHECK]“ Electrical features calculated: power_factor
[CHECK]“ Load categories assigned: 4 categories
[CHECK]“ Anomalies detected: 214 records
[CHECK]“ Rolling averages computed: 24h, 7d
[CHECK]“ Total features engineered: 29

Next: Proceed to EDA analysis notebook
